In [ ]:
# ===================================================================
#      JAX (Brax PPO) to ONNX Conversion Script (Complete & Final)
# ===================================================================

# --- 導入所有必要的函式庫 ---
import pickle
import numpy as np
import jax
import tensorflow as tf
from tensorflow.keras import layers
import tf2onnx
from etils import epath
import os

# 導入 Brax 相關模組以重建網路
from brax.training.acme import running_statistics
from brax.training.agents.ppo import networks as ppo_networks

# 為了載入 Orbax checkpoint
from orbax import checkpoint as ocp

# --- Step 0: 配置 (請根據您的訓練進行修改) ---

# 【請修改】指向您 PPO 訓練的 Checkpoint 主目錄
# 這應該是包含了 '100000', '2000000' 等子目錄的那個'checkpoints'目錄
CHECKPOINT_DIR = epath.Path("checkpoints/PupperJoystickFlatTerrain").resolve() 

# 【請修改】您 Pupper 環境的觀測和動作維度
POLICY_OBS_SIZE = 48 
ACTION_SIZE = 12

# 【請修改】您 PPO 訓練時使用的網路結構
POLICY_HIDDEN_LAYER_SIZES = (512, 256, 128)


# --- Step 1: 手動查找並使用 Orbax 載入最新的 JAX 參數 ---
print(f"--- Step 1: Loading JAX parameters from Orbax checkpoint ---")

# 【關鍵修改 1】: 在定義 CHECKPOINT_DIR 時，就將其轉換為絕對路徑
CHECKPOINT_DIR = epath.Path("checkpoints/PupperJoystickFlatTerrain").resolve()

print(f"Searching for checkpoints in: {CHECKPOINT_DIR}")

try:
    if not CHECKPOINT_DIR.exists() or not any(CHECKPOINT_DIR.iterdir()):
        raise FileNotFoundError(f"Checkpoint directory is empty or does not exist: {CHECKPOINT_DIR}")

    # 1. 查找最新的步數 (這部分邏輯正確)
    steps = [int(p.name) for p in CHECKPOINT_DIR.iterdir() if p.is_dir() and p.name.isdigit()]
    if not steps:
        raise FileNotFoundError(f"No valid checkpoint steps found in: {CHECKPOINT_DIR}")
    latest_step = max(steps)
    print(f"  - Found latest checkpoint at step: {latest_step}")

    # 2. 構建最新的 checkpoint 路徑 (現在它將是絕對路徑)
    latest_checkpoint_path = CHECKPOINT_DIR / str(latest_step)
    print(f"  - Restoring from: {latest_checkpoint_path}")

    # 3. 【關鍵修改 2】: 使用 Orbax 的標準方式來恢復
    #    創建一個 Checkpointer 實例，然後呼叫 restore
    checkpointer = ocp.PyTreeCheckpointer(str(latest_checkpoint_path))
    params_tuple = checkpointer.restore()
    
    # 分離參數
    normalizer_params, policy_params, value_params = params_tuple
    
    print("  - JAX parameters loaded and separated successfully.")

except FileNotFoundError as e:
    print(f"  [ERROR] {e}")
    raise
except Exception as e:
    print(f"  [ERROR] An unexpected error occurred while loading Orbax checkpoint: {e}")
    # raise e # 暫時不拋出異常，方便觀察


# --- 步驟 2: 創建一個包含正規化層的等效 TensorFlow Keras 模型 ---
print("\n--- Step 2: Building equivalent TensorFlow Keras model ---")

# 定義與 Brax MLP 結構對應的 Keras MLP 模型
class MLP(tf.keras.Model):
    def __init__(self, layer_sizes, activation=tf.nn.relu, name='mlp_block'):
        super().__init__(name=name)
        self.mlp_layers = []
        for i, size in enumerate(layer_sizes):
            act = activation if i < len(layer_sizes) - 1 else None
            self.mlp_layers.append(layers.Dense(
                size, activation=act, kernel_initializer='lecun_uniform', name=f"hidden_{i}"
            ))
    def call(self, inputs):
        x = inputs
        for layer in self.mlp_layers:
            x = layer(x)
        return x

# 定義工廠函式來創建完整的、端到端的策略網路
def make_tf_policy_network(obs_size, act_size, hidden_sizes, normalizer_mean, normalizer_std):
    inputs = tf.keras.Input(shape=(obs_size,), name='state')
    normalized_obs = (inputs - normalizer_mean) / (normalizer_std + 1e-8)
    mlp = MLP(layer_sizes=list(hidden_sizes) + [act_size * 2])
    logits = mlp(normalized_obs)
    loc, _ = tf.split(logits, num_or_size_splits=2, axis=-1)
    outputs = tf.keras.layers.Activation('tanh', name='action')(loc)
    return tf.keras.Model(inputs=inputs, outputs=outputs, name="PupperPPOPolicy")

# 【關鍵修正】: 使用點 `.` 來訪問 dataclass 屬性
mean = np.array(normalizer_params.mean['state'])
std = np.array(normalizer_params.std['state'])

# 創建 TF 模型
tf_policy_network = make_tf_policy_network(
    obs_size=POLICY_OBS_SIZE,
    act_size=ACTION_SIZE,
    hidden_sizes=POLICY_HIDDEN_LAYER_SIZES,
    normalizer_mean=mean,
    normalizer_std=std
)
print("  - TensorFlow Keras model with normalization layer built successfully.")
tf_policy_network.summary()


# --- 步驟 3: 將 JAX 權重手動轉移到 TensorFlow 模型 ---
print("\n--- Step 3: Transferring JAX weights to TensorFlow model ---")

def transfer_weights(jax_policy_params, tf_model):
    # Brax PPO 網路的權重儲存在 params['policy'] 中
    jax_weights_dict = jax_policy_params['params']
    tf_mlp_layers = tf_model.get_layer('mlp_block').layers
    
    i = 0
    for layer in tf_mlp_layers:
        if isinstance(layer, layers.Dense):
            layer_name_jax = f"hidden_{i}"
            
            if layer_name_jax not in jax_weights_dict:
                print(f"  [WARNING] Weight key '{layer_name_jax}' not found in JAX params. Skipping layer '{layer.name}'.")
                continue

            jax_layer_params = jax_weights_dict[layer_name_jax]
            kernel = np.array(jax_layer_params['kernel'])
            bias = np.array(jax_layer_params['bias'])
            print(f"  - Transferring to TF layer '{layer.name}': kernel{kernel.shape}, bias{bias.shape}")
            layer.set_weights([kernel, bias])
            i += 1

# 執行權重轉移
transfer_weights(policy_params, tf_policy_network)
print("  - Weight transfer complete.")


# --- 步驟 4: 將 TensorFlow 模型轉換為 ONNX ---
print("\n--- Step 4: Converting TensorFlow model to ONNX ---")
output_path = f"pupper_ppo_policy_{latest_step}_normalized.onnx"

spec = (tf.TensorSpec((None, POLICY_OBS_SIZE), tf.float32, name="state"),)

try:
    model_proto, _ = tf2onnx.convert.from_keras(
        tf_policy_network, 
        input_signature=spec, 
        opset=13,
        output_path=output_path
    )

    print(f"\nConversion successful! ONNX model saved to: {output_path}")

    print("Checking the ONNX model...")
    onnx.checker.check_model(output_path)
    print("ONNX model checked successfully.")

except Exception as e:
    print(f"  [ERROR] An error occurred during ONNX conversion: {e}")

INFO:absl:Created BasePyTreeCheckpointHandler: use_ocdbt=True, use_zarr3=False, pytree_metadata_options=PyTreeMetadataOptions(support_rich_types=False), array_metadata_store=<orbax.checkpoint._src.metadata.array_metadata_store.Store object at 0x77617e2599d0>


INFO:absl:Restoring checkpoint from checkpoints/PupperJoystickFlatTerrain/200540160.


--- Step 1: Loading JAX parameters from Orbax checkpoint ---
Searching for checkpoints in: /mnt/d/project_pupper/mujoco/mujoco_playground_recoil/mujoco_playground/_src/locomotion/pupper/checkpoints/PupperJoystickFlatTerrain
  - Found latest checkpoint at step: 200540160
  - Restoring from: checkpoints/PupperJoystickFlatTerrain/200540160
  [ERROR] An unexpected error occurred while loading Orbax checkpoint: Checkpoint path should be absolute. Got checkpoints/PupperJoystickFlatTerrain/200540160


/home/kiwi/miniconda3/envs/mujoco/lib/python3.12/site-packages/orbax/checkpoint/_src/serialization/type_handlers.py:1251: UserWarning: Sharding info not provided when restoring. Populating sharding info from sharding file. Please note restoration time will be slightly increased due to reading from file. Note also that this option is unsafe when restoring on a different topology than the checkpoint was saved with.
  warnings.warn(


ValueError: Checkpoint path should be absolute. Got checkpoints/PupperJoystickFlatTerrain/200540160